# AeroPulse Source Likelihood — 07: Temporal, Spatial & Event Evaluation

Event-based splits, geographic-block holdout, and per-source degradation —
the validations that decide whether this generalises beyond the rows it
was fitted on.

## Objective

Improvement plan sections 29, 30, 31 and 27.

Three questions, none of which a single chronological row split can answer:

- **section 30** — do the results hold when whole *events* are held out?
  Hours inside one episode are so correlated that a row split lets the same
  episode appear in train and test.
- **section 31** — do they hold in a region the model has never seen?
  A random station holdout leaks the local pollution climate; geographic
  blocks do not.
- **section 29** — does the model identify the dominant source *of an
  event*, which is the unit AeroPulse actually reports on?

Every number here still measures agreement with the weak labels, per
section 27. Notebook 08 is where that ceiling is confronted directly.


> **Scientific constraint, repeated in every notebook because it governs how
> the output may be used.** These are *likelihoods derived from heuristics*,
> not measured source attribution. No public ground-truth attribution
> catalogue exists for this dataset. A model that scores well here has
> reproduced its own labeling system — which is not the same as being right.

In [1]:
# ============================================================================
# SETUP — reusable logic lives in source_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import source_toolkit as st

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

cfg = st.load_config()
SOURCES = st.SOURCES
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood
sources: ['biomass_burning', 'traffic', 'industrial', 'dust', 'regional_transport']
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


In [2]:
import joblib
cal_stage = cfg.stage_dir("calibration")
clf_stage = cfg.stage_dir("classifiers")
cal_report = json.loads((cal_stage / "calibration_report.json").read_text())
clf_report = json.loads((clf_stage / "classifier_report.json").read_text())
WINNER = clf_report["winner_family"]
TRAINABLE = clf_report["trainable_sources"]
FEATURES = clf_report["features"]
BEST_METHOD = cal_report["selected_method"]
THRESHOLDS = cal_report["thresholds"]

df = pd.read_parquet(cfg.feature_dir / "source_evidence_features.parquet")
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
labels = pd.read_parquet(cfg.label_dir / "weak_labels.parquet")
work = df.join(labels)
events = pd.read_parquet(cfg.label_dir / "events.parquet")

def subsample(frame, cap):
    if cap is None or len(frame) <= cap:
        return frame
    return frame.iloc[(np.arange(cap) * (len(frame) / cap)).astype(int)]

CAP = cfg.max_train_rows or 400_000
print("family", WINNER, "| sources", TRAINABLE, "| features", len(FEATURES))

family lightgbm | sources ['biomass_burning', 'traffic', 'industrial', 'dust', 'regional_transport'] | features 56


## 1. Row-based versus event-based splitting (section 30)

In [3]:
row_train, row_valid, row_test = st.chrono_split(work, cfg.train_fraction, cfg.validation_fraction)
ev_train, ev_valid, ev_test = st.event_split(work, events, cfg.train_fraction, cfg.validation_fraction)

print(f"row split   : train {len(row_train):,}  valid {len(row_valid):,}  test {len(row_test):,}")
print(f"event split : train {len(ev_train):,}  valid {len(ev_valid):,}  test {len(ev_test):,}")
print("\nthe event split covers only rows inside an episode, which is the")
print("population AeroPulse reports on -- it is a different question, not a")
print("subset of the same one.")

def _keys(part):
    return set(part["location_id"].astype(str) + "::" + part["event_id"].astype(str))
overlap = _keys(ev_train) & _keys(ev_test)
print(f"\nevents straddling the event-split boundary: {len(overlap)}")
assert not overlap

row split   : train 1,193,612  valid 255,833  test 255,807
event split : train 199,939  valid 32,725  test 21,445

the event split covers only rows inside an episode, which is the
population AeroPulse reports on -- it is a different question, not a
subset of the same one.

events straddling the event-split boundary: 0


In [4]:
rows = []
for source in TRAINABLE:
    for name, (tr, te) in (("row_split", (subsample(row_train, CAP), row_test)),
                           ("event_split", (subsample(ev_train, CAP), ev_test))):
        if min(int(tr[f"y_{source}"].sum()), int((1 - tr[f"y_{source}"]).sum())) < 50:
            continue
        model = st.SourceClassifier.fit(tr, FEATURES, source, kind=WINNER)
        metrics = st.calibration_metrics(te[f"y_{source}"].to_numpy(), model.raw_proba(te))
        rows.append({"source": source, "split": name, "n_test": metrics["n"],
                     "prevalence": round(metrics["prevalence"], 4),
                     "pr_auc": round(metrics["pr_auc"], 4),
                     "pr_auc_lift": round(metrics["pr_auc_lift"], 3),
                     "roc_auc": round(metrics["roc_auc"], 4)})
split_compare = pd.DataFrame(rows)
print(split_compare.to_string(index=False))
print()
print(split_compare.pivot_table(index="source", columns="split", values="pr_auc_lift").round(3).to_string())

            source       split  n_test  prevalence  pr_auc  pr_auc_lift  roc_auc
   biomass_burning   row_split  255807      0.0007  0.0073       10.826   0.8507
   biomass_burning event_split   21445      0.4688  0.9137        1.949   0.9275
           traffic   row_split  255807      0.2145  0.6190        2.886   0.8099
           traffic event_split   21445      0.5669  0.8638        1.524   0.8384
        industrial   row_split  255807      0.0412  0.1446        3.511   0.7693
        industrial event_split   21445      0.1950  0.4811        2.467   0.8033
              dust   row_split  255807      0.3627  0.7160        1.974   0.8226
              dust event_split   21445      0.5100  0.9279        1.820   0.9067
regional_transport   row_split  255807      0.0046  0.0151        3.284   0.7830
regional_transport event_split   21445      0.0464  0.1351        2.913   0.7510

split               event_split  row_split
source                                    
biomass_burning       

## 2. Geographic-block holdout (section 31)

In [5]:
blocks = st.geographic_blocks(work, n_blocks=cfg.geographic_blocks, random_state=cfg.random_state)
print(blocks.groupby("geo_block").agg(stations=("location_id", "size"),
                                      lat=("block_lat", "first"),
                                      lon=("block_lon", "first")).round(2).to_string())
block_of = dict(zip(blocks["location_id"], blocks["geo_block"]))
work["geo_block"] = work["location_id"].map(block_of)

geo_rows = []
for source in TRAINABLE:
    for block in sorted(blocks["geo_block"].unique()):
        tr = subsample(work[work["geo_block"] != block], CAP)
        te = subsample(work[work["geo_block"] == block], 150_000)
        if min(int(tr[f"y_{source}"].sum()), int((1 - tr[f"y_{source}"]).sum())) < 50:
            continue
        if len(np.unique(te[f"y_{source}"])) < 2:
            geo_rows.append({"source": source, "held_block": int(block),
                             "stations": int(te["location_id"].nunique()),
                             "n": len(te), "prevalence": float(te[f"y_{source}"].mean()),
                             "pr_auc": np.nan, "pr_auc_lift": np.nan,
                             "note": "single-class in held-out block"})
            continue
        model = st.SourceClassifier.fit(tr, FEATURES, source, kind=WINNER)
        metrics = st.calibration_metrics(te[f"y_{source}"].to_numpy(), model.raw_proba(te))
        geo_rows.append({"source": source, "held_block": int(block),
                         "stations": int(te["location_id"].nunique()), "n": metrics["n"],
                         "prevalence": round(metrics["prevalence"], 4),
                         "pr_auc": round(metrics["pr_auc"], 4),
                         "pr_auc_lift": round(metrics["pr_auc_lift"], 3),
                         "note": ""})
    done = [r for r in geo_rows if r["source"] == source and np.isfinite(r.get("pr_auc_lift", np.nan))]
    if done:
        print(f"  {source:<20} lift by block: " +
              " ".join(f"{r['pr_auc_lift']:.2f}" for r in done))

geo = pd.DataFrame(geo_rows)
print()
print(geo.pivot_table(index="source", columns="held_block", values="pr_auc_lift").round(3).to_string())

           stations        lat        lon
geo_block                                
0                27  19.010000  79.169998
1                28  27.860001  77.500000
2                31  23.610001  86.830002
3                42  18.920000  73.660004
4                21  24.639999  74.230003


  biomass_burning      lift by block: 3.54 3.61 3.39 3.69 5.74


  traffic              lift by block: 1.93 1.85 1.97 2.06 2.05


  industrial           lift by block: 3.00 4.77 3.69 2.67 4.21


  dust                 lift by block: 2.62 1.45 1.64 1.76 1.98


  regional_transport   lift by block: 5.44 3.21 5.13 6.28 6.52

held_block              0      1      2      3      4
source                                               
biomass_burning     3.537  3.611  3.394  3.688  5.735
dust                2.617  1.445  1.635  1.758  1.984
industrial          3.005  4.772  3.695  2.666  4.214
regional_transport  5.444  3.214  5.131  6.280  6.522
traffic             1.929  1.855  1.970  2.060  2.049


## 3. Event-level source agreement (section 29)

In [6]:
models = {}
for source in TRAINABLE:
    tr = subsample(row_train, CAP)
    model = st.SourceClassifier.fit(tr, FEATURES, source, kind=WINNER)
    model.calibrate(row_valid, method=BEST_METHOD[source])
    model.threshold = THRESHOLDS[source]
    models[source] = model

scored = ev_test.copy()
for source in TRAINABLE:
    scored[f"p_{source}"] = models[source].predict_proba(scored)

prob_cols = [f"p_{s}" for s in TRAINABLE]
event_pred = (
    scored.groupby(["location_id", "event_id"], observed=True)[prob_cols]
    .mean().reset_index()
)
event_pred["model_dominant"] = [
    prob_cols[i].replace("p_", "") for i in event_pred[prob_cols].to_numpy().argmax(axis=1)
]
truth = events[["location_id", "event_id", "dominant_source", "n_hours", "peak_pm25"]]
merged = event_pred.merge(truth, on=["location_id", "event_id"], how="inner")

agreement = float((merged["model_dominant"] == merged["dominant_source"]).mean())
print(f"events scored: {len(merged):,}")
print(f"event-level dominant-source agreement with the weak label: {agreement:.1%}")
print(f"random-guess baseline (1/{len(TRAINABLE)}): {1 / len(TRAINABLE):.1%}")
majority = merged["dominant_source"].value_counts(normalize=True).iloc[0]
print(f"majority-class baseline: {majority:.1%}")
print("\nconfusion (rows = weak label, columns = model):")
print(pd.crosstab(merged["dominant_source"], merged["model_dominant"]).to_string())

events scored: 6,685
event-level dominant-source agreement with the weak label: 6.2%
random-guess baseline (1/5): 20.0%
majority-class baseline: 73.9%

confusion (rows = weak label, columns = model):
model_dominant      biomass_burning  dust  industrial  regional_transport  traffic
dominant_source                                                                   
biomass_burning                  16   175          22                   0      144
dust                              2    61           4                   0       68
industrial                       62  3125          53                   2     1701
regional_transport                9   233          31                   0      190
traffic                           6   475          19                   0      287


## 4. Per-source degradation summary

In [7]:
rows = []
for source in TRAINABLE:
    row_lift = split_compare[(split_compare["source"] == source) &
                             (split_compare["split"] == "row_split")]["pr_auc_lift"]
    ev_lift = split_compare[(split_compare["source"] == source) &
                            (split_compare["split"] == "event_split")]["pr_auc_lift"]
    block_lifts = geo[(geo["source"] == source)]["pr_auc_lift"].dropna()
    rows.append({
        "source": source,
        "row_split_lift": float(row_lift.iloc[0]) if len(row_lift) else np.nan,
        "event_split_lift": float(ev_lift.iloc[0]) if len(ev_lift) else np.nan,
        "geo_min_lift": round(float(block_lifts.min()), 3) if len(block_lifts) else np.nan,
        "geo_mean_lift": round(float(block_lifts.mean()), 3) if len(block_lifts) else np.nan,
        "geo_blocks_scored": int(len(block_lifts)),
    })
degradation = pd.DataFrame(rows)
print(degradation.to_string(index=False))

            source  row_split_lift  event_split_lift  geo_min_lift  geo_mean_lift  geo_blocks_scored
   biomass_burning          10.826             1.949         3.394          3.993                  5
           traffic           2.886             1.524         1.855          1.973                  5
        industrial           3.511             2.467         2.666          3.670                  5
              dust           1.974             1.820         1.445          1.888                  5
regional_transport           3.284             2.913         3.214          5.318                  5


## 5. Save

In [8]:
stage = cfg.stage_dir("evaluation")
split_compare.to_csv(stage / "split_comparison.csv", index=False)
geo.to_csv(stage / "geographic_blocks.csv", index=False)
blocks.to_csv(stage / "station_blocks.csv", index=False)
merged.to_csv(stage / "event_predictions.csv", index=False)
degradation.to_csv(stage / "degradation_summary.csv", index=False)

report = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "split_comparison": split_compare.to_dict(orient="records"),
    "geographic": geo.to_dict(orient="records"),
    "event_agreement": agreement,
    "event_majority_baseline": float(majority),
    "n_events_scored": int(len(merged)),
    "degradation": degradation.to_dict(orient="records"),
    "caveat": "agreement with weak labels, not attribution accuracy",
}
(stage / "evaluation_report.json").write_text(json.dumps(report, indent=2, default=str))
print("07 COMPLETE ->", stage)
print("Next: 08_source_gold_set_evaluation.ipynb")

07 COMPLETE -> /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/source_likelihood/artifacts/source/evaluation
Next: 08_source_gold_set_evaluation.ipynb


## Findings

**Geographic generalisation holds, which was not the expected result.**
Notebook 05 found the classifiers key overwhelmingly on `lat`, `lon` and
day-of-year, which predicts collapse on an unseen region. It does not
happen — every source keeps positive lift on all five held-out blocks:

| source | min block lift | mean |
|---|---|---|
| regional_transport | 3.21x | 5.32x |
| biomass_burning | 3.39x | 3.99x |
| industrial | 2.67x | 3.67x |
| traffic | 1.86x | 1.97x |
| dust | **1.45x** | 1.89x |

Dust at 1.45x is the weakest cell and the only one below the 1.5x spatial
gate. The reconciliation with notebook 05 is that coordinates are being used
as a *proxy for regional climate* rather than as station identity — the
model transfers to a new block because that block's climate is still
interpolable from the others. It would not survive a genuinely different
airshed.

**Event-based splitting costs a lot, and it is the honest number.**

| source | row split | event split |
|---|---|---|
| biomass_burning | 10.83x | **1.95x** |
| industrial | 3.51x | 2.47x |
| traffic | 2.89x | 1.52x |
| regional_transport | 3.28x | 2.91x |
| dust | 1.97x | 1.82x |

Biomass burning loses 82% of its apparent lift once whole episodes are held
out rather than individual hours. Hours inside one episode are near
duplicates, so a row split lets the model see most of an event's evidence
before being asked about the rest of it. Plan section 30 predicted this;
the size of the drop is what makes it worth enforcing.

**Event-level agreement is 6.2%, and that number needs care.** Against a
20% random baseline and a 73.9% majority-class baseline, the model's
event-level dominant source agrees with the weak label's on 6.2% of 6,685
events — worse than guessing.

Before concluding the model has failed, look at what is being compared. The
confusion matrix shows the weak label says `industrial` on 4,943 events
while the model says `dust` or `traffic`. Notebook 04 already flagged why:
the weak label's event-level dominance averages **NaN-sparse** probabilities,
so a source evaluated on three hours out of forty can win on mean, and
`industrial` is evaluated on only 13.2% of rows. The model's event
prediction averages dense calibrated probabilities over every hour.

**The two sides are aggregating different quantities, so 6.2% measures a
metric mismatch as much as a model failure.** The fix is to make event-level
weak dominance coverage-weighted before scoring against it. Until that is
done this figure should not be quoted as event accuracy in either direction
— and it is exactly the kind of number that would have been quoted as a
headline had the aggregation not been checked.